# 喜马拉雅 Colab Worker（GitHub 版 · CF Worker 中转 TG）

代码从 GitHub 自动拉取，无需手动上传。TG 上传通过 Cloudflare Worker 中转代理，喜马拉雅下载直连本机 IP（不走代理）。

## 特性
- 自动从 GitHub 克隆最新代码
- 支持 `--num-workers` 多线程并行处理
- tqdm 实时进度条
- 所有配置通过 VPS 全局设置下发（TG Token / Cookie / 代理 / 降噪模型等）
- **TG 上传走 CF Worker 中转**（设置 `TELEGRAM_API_BASE` 环境变量）
- **喜马拉雅下载不走代理，直连本机 IP**（`--no-proxy`）

> **使用方法**：修改下方配置后，从上到下依次执行每个 Cell。

## 1. 配置

修改下方参数后执行：

In [ ]:
# ═══ VPS 连接 ═══
VPS_URL = "http://117.55.234.219:59388"
WORKER_ID = "colab_mt_cf_001"      # 留空则自动生成
WORKER_TOKEN = "gchqRbJR2kmop5YwQQeYNGLs"

# ═══ Cloudflare Worker 中转 ═══
# CF Worker 代理地址（不要带末尾 /）
# 留空则直连 Telegram API（与原版行为一致）
CF_WORKER_URL = "https://ximalaya.supersolar520.xyz"

# ═══ 多线程 ═══
import os
NUM_WORKERS = os.cpu_count() or 2  # 根据 CPU 核心数自动设置

# ═══ GitHub 仓库 ═══
GIT_REPO = "https://github.com/collinsgraciano/ximalaya_manager.git"
GIT_BRANCH = "main"               # 分支名

# ═══ 运行参数 ═══
POLL_INTERVAL = 10                # 无任务时等待秒数
MAX_JOBS = 0                      # 最大处理任务数 (0=不限)

print(f"配置完成 | CPU: {os.cpu_count()} 核 | 线程数: {NUM_WORKERS}")
if CF_WORKER_URL:
    print(f"TG 中转: {CF_WORKER_URL}")
else:
    print("TG 中转: 直连 (未配置 CF Worker)")

## 2. 克隆代码 & 安装依赖

每次运行都从 GitHub 拉取最新代码。

In [ ]:
!rm -rf /content/ximalaya_manager
!git clone -b {GIT_BRANCH} {GIT_REPO} /content/ximalaya_manager

# 安装依赖
!pip install -q requests pycryptodome tqdm pydub
!apt-get -qq install -y ffmpeg

print("代码克隆 & 依赖安装完成")

## 3. 测试 CF Worker 中转

验证 CF Worker 能否正常代理 Telegram Bot API 请求。配置了 `CF_WORKER_URL` 时执行，直连模式可跳过。

In [ ]:
import requests

if not CF_WORKER_URL:
    print("未配置 CF_WORKER_URL，跳过测试（直连模式）")
else:
    base = CF_WORKER_URL.rstrip("/")

    # 从 VPS 获取第一个 Bot Token 用于测试
    print("从 VPS 获取 Bot Token...")
    cfg_resp = requests.get(
        f"{VPS_URL}/api/config",
        params={"worker_id": WORKER_ID or "cf_test", "worker_token": WORKER_TOKEN},
        timeout=30,
    ).json()

    tokens = cfg_resp.get("config", {}).get("tg_bot_tokens", [])
    if not tokens:
        print("⚠️ VPS 未配置 Bot Token，跳过测试")
    else:
        token = tokens[0]
        url = f"{base}/bot{token}/getMe"
        print(f"测试: GET {base}/bot***/getMe")
        r = requests.get(url, timeout=30)
        print(f"HTTP {r.status_code}")
        result = r.json()
        if result.get("ok"):
            bot_name = result["result"]["username"]
            print(f"✅ CF Worker 中转正常 | Bot: @{bot_name}")
        else:
            print(f"❌ CF Worker 返回错误: {result}")
            raise Exception("CF Worker 测试失败，请检查配置")

## 4. 启动 Worker

执行后开始轮询任务。通过 `TELEGRAM_API_BASE` 环境变量让 `tg_upload.py` 走 CF Worker 中转。

In [ ]:
import os, subprocess, sys

worker_id = WORKER_ID or f"colab_mt_cf_{os.urandom(4).hex()}"

cmd = [
    sys.executable, "/content/ximalaya_manager/colab/ximalaya_colab_worker.py",
    "--vps-url", VPS_URL,
    "--worker-id", worker_id,
    "--worker-token", WORKER_TOKEN,
    "--num-workers", str(NUM_WORKERS),
    "--poll-interval", str(POLL_INTERVAL),
    "--max-jobs", str(MAX_JOBS),
    "--install-deps",
    "--no-proxy",
]

# 注入环境变量：CF Worker 中转 TG API
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
if CF_WORKER_URL:
    env["TELEGRAM_API_BASE"] = CF_WORKER_URL.rstrip("/")
    print(f"TG 中转已启用: {CF_WORKER_URL}")
else:
    print("TG 直连模式（未设置 TELEGRAM_API_BASE）")

print(f"启动: {' '.join(cmd)}", flush=True)

# Popen + PIPE: 逐行读取子进程输出，通过 print 转发到 Colab ipykernel
proc = subprocess.Popen(
    cmd, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nWorker 退出，返回码: {proc.returncode}", flush=True)